# XRD Pattern Visualization (SP only)

SP データのみを、角度補正せずに可視化して `../0_Data/3_Figure/XRD_Pattern_SP` に保存する notebook。

In [1]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MultipleLocator

from pymatgen.io.cif import CifParser
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from sklearn.preprocessing import MinMaxScaler

plt.rcParams["font.size"] = 25
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS"]


In [2]:
# データが含まれるフォルダーのパスを定義
folder_path = "../0_Data/2_Compiled_Data/"
Sample_name_list = pd.read_excel(folder_path + "Sample_name_list.xlsx")

# SPring-8 のリートベルトデータフォルダーのみを使用
M_S_path = "../0_Data/1_Rietveld/SPring-8/Mafic"
P_S_path = "../0_Data/1_Rietveld/SPring-8/Pelitic"
path_rietveld_folder = [M_S_path, P_S_path]

# 指定されたディレクトリおよびそのサブディレクトリ内のすべての .xy ファイルを見つける
full_path_glob = []
for path in path_rietveld_folder:
    full_path_glob.extend(glob.glob(path + "/*/*.xy"))

# macOS の AppleDouble ファイルを除外
full_path_glob = [path for path in full_path_glob if not os.path.basename(path).startswith("._")]

Path_Df = pd.DataFrame({"xy": [os.path.basename(path) for path in full_path_glob], "Path": full_path_glob})
Path_Df.index = Path_Df["xy"].str.rsplit(".", n=1).str[0]

name_SP_Mafic = Sample_name_list[Sample_name_list["Rock_Type"] == "Mafic Schist"]["SP-8"].dropna()
name_SP_Pelitic = Sample_name_list[Sample_name_list["Rock_Type"] == "Pelitic Schist"]["SP-8"].dropna()
name_SP = Sample_name_list["SP-8"].dropna()

lambda_SP = 0.6999200
fig_folder_path = "../0_Data/3_Figure/XRD_Pattern_SP"
os.makedirs(fig_folder_path, exist_ok=True)


In [3]:
def two_theta_to_d_value(two_theta, wavelength):
    """Convert 2theta angles to d-values without changing the original angle scale."""
    theta_radians = np.radians(np.asarray(two_theta, dtype=float) / 2)
    return wavelength / (2 * np.sin(theta_radians))


def read_sp_pattern(sample_name):
    path = Path_Df.loc[sample_name, "Path"]
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df.columns = ["2θ", "Intensity"]
    df["d-value"] = two_theta_to_d_value(df["2θ"], lambda_SP)
    return df


def minmax_intensity(df):
    df = df.copy()
    scaler = MinMaxScaler()
    df["Intensity"] = scaler.fit_transform(df[["Intensity"]]).ravel()
    return df


# CIF reference patterns

GL と Actinolite の CIF から、SP 波長でシミュレーションした参照ピークを作成します。

In [4]:
gl_cif = "../0_Data/0_Single_Crystal_XRD/240122/GL_noH_542_check_VESTA_240122_Modified.cif"
act_cif = "../0_Data/0_Single_Crystal_XRD/240122/Actinolite_0001986.cif"


def read_cif_xrd_pattern(cif_path, wavelength):
    parser = CifParser(cif_path)
    structure = parser.parse_structures(primitive=False, symmetrized=False)[0]
    pattern = XRDCalculator(wavelength=wavelength).get_pattern(structure)

    df = pd.DataFrame({"2θ": pattern.x, "Intensity": pattern.y})
    df = minmax_intensity(df)
    df["d-value"] = two_theta_to_d_value(df["2θ"], wavelength)
    return df


xrd_data_gl_df = read_cif_xrd_pattern(gl_cif, lambda_SP)
xrd_data_act_df = read_cif_xrd_pattern(act_cif, lambda_SP)


/Users/satoshimatsuno/Library/Caches/pypoetry/virtualenvs/pxrd-validation-ms--qvz1pt9-py3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1314: UserWarning: Missing elements H from PMG structure composition
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


# Visualize SP Diffraction Pattern

In [11]:
# 2θ limitation: SP の角度をそのまま使用
# limit_value_list = [[4, 67], [4, 67], [18, 20], [30, 35], [25, 30], [9.1, 11.4], [27.5, 28.5]]
limit_value_list = [
    [2, 40],
    [8.15, 9.05],
    [11.60, 11.75],
    
    [13.5, 15.7],
    [11.2, 13.5],
    [4.1, 5.2],
    [12.4, 12.85],
    [8.35, 8.85],
    
]

limit_value_list_wide = limit_value_list[0]


In [12]:
n = 0
for limit_value in limit_value_list:
    for SP_sample_name in name_SP:
        print(SP_sample_name)
        sample_title = SP_sample_name

        df_SP = read_sp_pattern(SP_sample_name)
        df_SP_filtered = df_SP[(df_SP["2θ"] >= limit_value[0]) & (df_SP["2θ"] <= limit_value[1])].copy()
        xrd_data_gl_df_filtered = xrd_data_gl_df[(xrd_data_gl_df["2θ"] >= limit_value[0]) & (xrd_data_gl_df["2θ"] <= limit_value[1])]
        xrd_data_act_df_filtered = xrd_data_act_df[(xrd_data_act_df["2θ"] >= limit_value[0]) & (xrd_data_act_df["2θ"] <= limit_value[1])]

        if df_SP_filtered.empty:
            print(f"Skip {SP_sample_name}: no SP data in {limit_value}")
            continue

        df_SP_filtered = minmax_intensity(df_SP_filtered)

        if limit_value == limit_value_list_wide:
            plt.figure(figsize=(30, 10))
            linewidth_plot_pattern = 3.5
        else:
            plt.figure(figsize=(13, 16))
            linewidth_plot_pattern = 6

        if limit_value != limit_value_list_wide:
            plt.vlines(
                xrd_data_gl_df_filtered["2θ"].values,
                0,
                xrd_data_gl_df_filtered["Intensity"].values,
                colors="#000080",
                linestyles="dashed",
                label="Sodium Amphibole",
                linewidth=linewidth_plot_pattern,
                zorder=4,
                alpha=0.5,
            )
            plt.vlines(
                xrd_data_act_df_filtered["2θ"].values,
                0,
                xrd_data_act_df_filtered["Intensity"].values,
                colors="#228B22",
                linestyles="dashed",
                label="Actinolite",
                linewidth=linewidth_plot_pattern,
                zorder=4,
                alpha=0.5,
            )

        plt.plot(
            df_SP_filtered["2θ"].values,
            df_SP_filtered["Intensity"].values,
            label="Synchrotron PXRD",
            color="#FF7F50",
            linewidth=linewidth_plot_pattern,
            zorder=3,
        )

        plt.xlabel("Diffraction Angle (°2θ, $\\lambda = 0.69992\\,\\AA$)", fontsize=40, labelpad=10)
        plt.ylabel("Intensity (arb. unit)", fontsize=40, labelpad=10)
        plt.title(SP_sample_name, fontsize=40, pad=20)

        if limit_value == limit_value_list_wide:
            ticks = np.arange(0, int(max(df_SP["2θ"].values)) + 5, 5)
            plt.xticks(ticks, fontsize=25)
            ax = plt.gca()
            ax.tick_params(axis="x", which="minor", direction='in', length=5, width=1.5)
            ax.tick_params(axis="x", which="major", direction='in', length=10, width=3)
        else:
            ax = plt.gca()
            x_min, x_max = sorted(limit_value)
            major_tick_interval = 0.5 if (x_max - x_min) > 0.8 else 0.1
            ax.xaxis.set_major_locator(MultipleLocator(major_tick_interval))
            ax.xaxis.set_minor_locator(MultipleLocator(0.1))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
            ax.tick_params(axis="x", which="minor", direction='in', length=5, width=1.5)
            ax.tick_params(axis="x", which="major", direction='in', length=10, width=3)
            plt.xticks(fontsize=25)

        plt.yticks([])
        plt.xlim(limit_value)

        if limit_value == limit_value_list_wide:
            plt.legend(fontsize=30, loc="upper right", frameon=False)

        plt.savefig(f"{fig_folder_path}/{sample_title}_{limit_value}_SP.pdf", bbox_inches="tight")
        plt.close()

        n += 1

print(n)


SP-MTN-42
SP-MTN-43
SP-MTN-44
SP-MTN-45
SP-MTN-46
SP-MTN-47
SP-MTN-56
SP-MTN-57
SP-MTN-58
SP-MTN-59
SP-MTN-60
SP-MTN-61
SP-MTN-70
SP-MTN-71
SP-MTN-72
SP-MTN-78
SP-MTN-83
SP-MTN-84
SP-MTN-85
SP-MTN-86
SP-MTN-87
SP-MTN-88
SP-MTN-92
SP-MTN-93
SP-MTN-94
SP-MTN-95
SP-MTN-41
SP-MTN-62
SP-MTN-63
SP-MTN-64
SP-MTN-65
SP-MTN-66
SP-MTN-67
SP-MTN-68
SP-MTN-69
SP-MTN-73
SP-MTN-74
SP-MTN-75
SP-MTN-76
SP-MTN-77
SP-MTN-79
SP-MTN-80
SP-MTN-89
SP-MTN-90
SP-MTN-96
SP-MTN-97
SP-MTN-98
SP-MTN-99
SP-MTN-42
SP-MTN-43
SP-MTN-44
SP-MTN-45
SP-MTN-46
SP-MTN-47
SP-MTN-56
SP-MTN-57
SP-MTN-58
SP-MTN-59
SP-MTN-60
SP-MTN-61
SP-MTN-70
SP-MTN-71
SP-MTN-72
SP-MTN-78
SP-MTN-83
SP-MTN-84
SP-MTN-85
SP-MTN-86
SP-MTN-87
SP-MTN-88
SP-MTN-92
SP-MTN-93
SP-MTN-94
SP-MTN-95
SP-MTN-41
SP-MTN-62
SP-MTN-63
SP-MTN-64
SP-MTN-65
SP-MTN-66
SP-MTN-67
SP-MTN-68
SP-MTN-69
SP-MTN-73
SP-MTN-74
SP-MTN-75
SP-MTN-76
SP-MTN-77
SP-MTN-79
SP-MTN-80
SP-MTN-89
SP-MTN-90
SP-MTN-96
SP-MTN-97
SP-MTN-98
SP-MTN-99
SP-MTN-42
SP-MTN-43
SP-MTN-44
SP-MTN-45


# Visualize SP d-value Pattern

In [9]:
# d-value limitation
limit_value_list = [[26, 1.5], [10, 8.3], [3.28, 3.18], [8.35, 8.85],]
limit_value_list_wide = limit_value_list[0]


In [10]:
n = 0
for limit_value in limit_value_list:
    for SP_sample_name in name_SP:
        print(SP_sample_name)
        sample_title = SP_sample_name

        df_SP = read_sp_pattern(SP_sample_name)
        df_SP_filtered = df_SP[(df_SP["d-value"] >= limit_value[1]) & (df_SP["d-value"] <= limit_value[0])].copy()
        xrd_data_gl_df_filtered = xrd_data_gl_df[(xrd_data_gl_df["d-value"] >= limit_value[1]) & (xrd_data_gl_df["d-value"] <= limit_value[0])]
        xrd_data_act_df_filtered = xrd_data_act_df[(xrd_data_act_df["d-value"] >= limit_value[1]) & (xrd_data_act_df["d-value"] <= limit_value[0])]

        if df_SP_filtered.empty:
            print(f"Skip {SP_sample_name}: no SP data in {limit_value}")
            continue

        df_SP_filtered = minmax_intensity(df_SP_filtered)

        if limit_value == limit_value_list_wide:
            plt.figure(figsize=(30, 10))
            linewidth_plot_pattern = 3.5
        else:
            plt.figure(figsize=(13, 16))
            linewidth_plot_pattern = 6

        if limit_value != limit_value_list_wide:
            plt.vlines(
                xrd_data_gl_df_filtered["d-value"].values,
                0,
                xrd_data_gl_df_filtered["Intensity"].values,
                colors="#000080",
                linestyles="dashed",
                label="Sodium Amphibole",
                linewidth=linewidth_plot_pattern,
                zorder=4,
                alpha=0.5,
            )
            plt.vlines(
                xrd_data_act_df_filtered["d-value"].values,
                0,
                xrd_data_act_df_filtered["Intensity"].values,
                colors="#228B22",
                linestyles="dashed",
                label="Actinolite",
                linewidth=linewidth_plot_pattern,
                zorder=4,
                alpha=0.5,
            )

        plt.plot(
            df_SP_filtered["d-value"].values,
            df_SP_filtered["Intensity"].values,
            label="Synchrotron PXRD",
            color="#FF7F50",
            linewidth=linewidth_plot_pattern,
            zorder=3,
        )

        plt.xlabel("d-value [Å]", fontsize=40, labelpad=10)
        plt.ylabel("Intensity (arb. unit)", fontsize=40, labelpad=10)
        plt.title(SP_sample_name, fontsize=40, pad=20)

        if limit_value == limit_value_list_wide:
            ticks = np.arange(0, int(max(df_SP["d-value"].values)) + 5, 5)
            plt.xticks(ticks, fontsize=25)
        else:
            ax = plt.gca()
            x_min, x_max = sorted(limit_value)
            major_tick_interval = 0.5 if (x_max - x_min) > 0.8 else 0.1
            ax.xaxis.set_major_locator(MultipleLocator(major_tick_interval))
            ax.xaxis.set_minor_locator(MultipleLocator(0.1))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
            ax.tick_params(axis="x", direction='in', which="minor", length=5, width=1.5)
            plt.xticks(fontsize=25)
        plt.yticks([])
        plt.xlim(limit_value)

        if limit_value == limit_value_list_wide:
            plt.legend(fontsize=30, loc="upper right", frameon=False)

        plt.gca().invert_xaxis()
        plt.savefig(f"{fig_folder_path}/{sample_title}_d-value_{limit_value}_SP.pdf", bbox_inches="tight")
        plt.close()

        n += 1

print(n)


SP-MTN-42
SP-MTN-43
SP-MTN-44
SP-MTN-45
SP-MTN-46
SP-MTN-47
SP-MTN-56
SP-MTN-57
SP-MTN-58
SP-MTN-59
SP-MTN-60
SP-MTN-61
SP-MTN-70
SP-MTN-71
SP-MTN-72
SP-MTN-78
SP-MTN-83
SP-MTN-84
SP-MTN-85
SP-MTN-86
SP-MTN-87
SP-MTN-88
SP-MTN-92
SP-MTN-93
SP-MTN-94
SP-MTN-95
SP-MTN-41
SP-MTN-62
SP-MTN-63
SP-MTN-64
SP-MTN-65
SP-MTN-66
SP-MTN-67
SP-MTN-68
SP-MTN-69
SP-MTN-73
SP-MTN-74
SP-MTN-75
SP-MTN-76
SP-MTN-77
SP-MTN-79
SP-MTN-80
SP-MTN-89
SP-MTN-90
SP-MTN-96
SP-MTN-97
SP-MTN-98
SP-MTN-99
SP-MTN-42
SP-MTN-43
SP-MTN-44
SP-MTN-45
SP-MTN-46
SP-MTN-47
SP-MTN-56
SP-MTN-57
SP-MTN-58
SP-MTN-59
SP-MTN-60
SP-MTN-61
SP-MTN-70
SP-MTN-71
SP-MTN-72
SP-MTN-78
SP-MTN-83
SP-MTN-84
SP-MTN-85
SP-MTN-86
SP-MTN-87
SP-MTN-88
SP-MTN-92
SP-MTN-93
SP-MTN-94
SP-MTN-95
SP-MTN-41
SP-MTN-62
SP-MTN-63
SP-MTN-64
SP-MTN-65
SP-MTN-66
SP-MTN-67
SP-MTN-68
SP-MTN-69
SP-MTN-73
SP-MTN-74
SP-MTN-75
SP-MTN-76
SP-MTN-77
SP-MTN-79
SP-MTN-80
SP-MTN-89
SP-MTN-90
SP-MTN-96
SP-MTN-97
Skip SP-MTN-97: no SP data in [10, 8.3]
SP-MTN-98
SP-MTN-99
